In [1]:
pip install requests lxml

Note: you may need to restart the kernel to use updated packages.


In [19]:
import requests
import sqlite3
import xml.etree.ElementTree as ET
import time

In [ ]:
API_KEY = "Enter_Your_API_Key_Here"

In [4]:
DB_NAME = 'patents.db'

In [12]:
def setup_database():
    """DB 및 테이블 생성: 청구항, 설명, 도면URL 컬럼 추가"""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS patent_fulltext (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            application_number TEXT,
            title TEXT,
            claims TEXT,
            description TEXT,
            drawings TEXT
        )
    ''')
    conn.commit()
    return conn

In [13]:
def get_target_application_numbers(limit=5):
    """1단계: 일반 검색 API를 통해 G06N 특허 출원번호 5개 가져오기"""
    url = "http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getWordSearch"
    
    # IPC 코드가 G06N인 특허를 검색어로 지정 (KIPRIS 검색 문법 적용)
    params = {
        'word': 'IPC=[G06N]', 
        'year': '0',
        'numOfRows': limit,
        'pageNo': 1,
        'ServiceKey': API_KEY
    }

    try:
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("검색 API 호출 실패")
            return []

        root = ET.fromstring(response.text)
        app_numbers = []
        
        # item 태그 안의 applicationNumber 추출
        for item in root.findall('.//item'):
            num = item.findtext('applicationNumber')
            if num:
                app_numbers.append(num)
                
        print(f"[{len(app_numbers)}개]의 출원번호를 확보했습니다: {app_numbers}")
        return app_numbers
    
    except Exception as e:
        print(f"검색 API 에러: {e}")
        return []

In [14]:
def fetch_full_text_and_save(app_num, conn):
    """2단계: 전문조회 API를 통해 명세서 전체 및 도면 정보 저장"""
    url = "http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getPubFullTextInfoSearch"
    
    params = {
        'applicationNumber': app_num,
        'ServiceKey': API_KEY
    }

    try:
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"[{app_num}] 전문 API 호출 실패")
            return

        root = ET.fromstring(response.text)
        
        # 정상 응답인지 체크
        if root.findtext('.//successYN') != 'Y':
            print(f"[{app_num}] API 응답 오류: {root.findtext('.//resultMsg')}")
            return

        item = root.find('.//item')
        if item is None:
            print(f"[{app_num}] 전문 데이터가 없습니다.")
            return

        # 1. 발명의 명칭
        title = item.findtext('.//inventionTitle') or "N/A"

        # 2. 청구항 (여러 개일 수 있으므로 하나로 합침)
        claims = []
        for claim_info in item.findall('.//claimInfo'):
            claim_text = claim_info.findtext('claim')
            if claim_text:
                claims.append(claim_text.strip())
        claims_str = "\n".join(claims)

        # 3. 발명의 설명 (상세한 설명)
        description = item.findtext('.//description') or item.findtext('.//detailDescCont') or "N/A"

        # 4. 도면 정보 (주로 이미지 URL 형태로 제공됨)
        drawings = []
        for img_info in item.findall('.//imagePathInfo'):
            path = img_info.findtext('path') or img_info.findtext('largePath')
            if path:
                drawings.append(path)
        drawings_str = "\n".join(drawings)

        # DB 저장
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO patent_fulltext (application_number, title, claims, description, drawings)
            VALUES (?, ?, ?, ?, ?)
        ''', (app_num, title, claims_str, description, drawings_str))
        conn.commit()
        
        print(f"[{app_num}] 파싱 및 저장 완료! (도면 {len(drawings)}개 포함)")

    except Exception as e:
        print(f"[{app_num}] 에러 발생: {e}")

In [15]:
db_conn = setup_database()

In [16]:
target_numbers = get_target_application_numbers(limit=5)

[5개]의 출원번호를 확보했습니다: ['1020227003979', '1020220151974', '1020257023914', '1020227040957', '1020230090053']


In [20]:
if target_numbers:
    print("\n각 출원번호의 명세서 전체 데이터를 가져옵니다...")
    for num in target_numbers:
        fetch_full_text_and_save(num, db_conn)
        time.sleep(1) # 서버 과부하 방지용 딜레이 (1초)
    else:
        print("검색된 출원번호가 없어 전문 조회를 종료합니다.")


각 출원번호의 명세서 전체 데이터를 가져옵니다...
[1020227003979] 파싱 및 저장 완료! (도면 0개 포함)
[1020220151974] 파싱 및 저장 완료! (도면 0개 포함)
[1020257023914] 파싱 및 저장 완료! (도면 0개 포함)
[1020227040957] 파싱 및 저장 완료! (도면 0개 포함)
[1020230090053] 파싱 및 저장 완료! (도면 0개 포함)
검색된 출원번호가 없어 전문 조회를 종료합니다.


In [21]:
import sqlite3
import pandas as pd

In [22]:
conn = sqlite3.connect('patents.db')

In [23]:
query = "SELECT * FROM patent_fulltext"
df = pd.read_sql_query(query, conn)

In [24]:
df

,id,application_number,title,claims,description,drawings
0,1,1020227003979,N/A,,N/A,
1,2,1020227003979,N/A,,N/A,
2,3,1020220151974,N/A,,N/A,
3,4,1020257023914,N/A,,N/A,
4,5,1020227040957,N/A,,N/A,
5,6,1020230090053,N/A,,N/A,


In [28]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time

# 표시되는 텍스트가 잘리지 않도록 Pandas 출력 설정 변경
# pd.set_option('display.max_colwidth', None)

# --- 설정 영역 ---
# 발급받으신 실제 KIPRIS 서비스 키로 반드시 변경해주세요.
API_KEY = 'jDv3cl1QtZCqenGSJiszX3BLW4xnviF5kVY0R8=R=c8='

# 조회할 출원번호 리스트
APP_NUMBERS = [
    '1020227003979',
    '1020220151974',
    '1020257023914',
    '1020227040957',
    '1020230090053'
]

In [26]:
def fetch_patent_details(app_num, api_key):
    """특정 출원번호의 서지/상세 정보를 가져와 딕셔너리로 반환합니다."""
    url = "http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getBibliographyDetailInfoSearch"
    params = {
        'applicationNumber': app_num,
        'ServiceKey': api_key
    }
    
    try:
        response = requests.get(url, params=params)
        if response.status_code != 200:
            return {"출원번호": app_num, "에러": f"HTTP {response.status_code}"}
            
        root = ET.fromstring(response.text)
        
        # 정상 응답 확인
        if root.findtext('.//successYN') != 'Y':
            return {"출원번호": app_num, "에러": root.findtext('.//resultMsg')}
            
        item = root.find('.//item')
        if item is None:
            return {"출원번호": app_num, "에러": "데이터 없음"}
            
        # 1. 출원번호 & 발명의 명칭
        extracted_app_num = item.findtext('.//applicationNumber') or app_num
        title = item.findtext('.//inventionTitle') or ""
        
        # 2. 출원인 (여러 명일 경우 콤마로 연결)
        applicants = [app.findtext('name') for app in item.findall('.//applicantInfo') if app.findtext('name')]
        applicant_str = ", ".join(applicants)
        
        # 3. IPC 코드 (여러 개일 경우 콤마로 연결)
        ipcs = [ipc.findtext('ipcNumber') for ipc in item.findall('.//ipcInfo') if ipc.findtext('ipcNumber')]
        ipc_str = ", ".join(ipcs)
        
        # 4. 청구항 (여러 개일 경우 줄바꿈으로 연결)
        claims = [claim.findtext('claim') for claim in item.findall('.//claimInfo') if claim.findtext('claim')]
        claim_str = "\n".join(claims)
        
        # 5. 도면 URL (largePath 우선, 없으면 path)
        drawings = []
        for img in item.findall('.//imagePathInfo'):
            path = img.findtext('largePath') or img.findtext('path')
            if path:
                drawings.append(path)
        drawing_str = "\n".join(drawings)
        
        return {
            "출원번호": extracted_app_num,
            "출원인": applicant_str,
            "IPC코드": ipc_str,
            "발명의 명칭": title,
            "청구항": claim_str,
            "도면URL": drawing_str
        }
        
    except Exception as e:
        return {"출원번호": app_num, "에러": str(e)}

In [29]:
# 결과를 담을 빈 리스트
results = []

print("데이터 수집을 시작합니다...")

for num in APP_NUMBERS:
    print(f"[{num}] 조회 중...")
    data = fetch_patent_details(num, API_KEY)
    results.append(data)
    time.sleep(0.5)  # API 서버 부하를 막기 위해 0.5초 대기

print("데이터 수집 완료!\n")

# 리스트를 Pandas DataFrame으로 변환
df = pd.DataFrame(results)

# DataFrame 출력 (Jupyter Lab에서는 변수명만 적으면 예쁜 표로 나옵니다)
df

데이터 수집을 시작합니다...
[1020227003979] 조회 중...
[1020220151974] 조회 중...
[1020257023914] 조회 중...
[1020227040957] 조회 중...
[1020230090053] 조회 중...
데이터 수집 완료!



출원번호                        출원인  \
0  10-2022-7003979                올리 펫츠 아이엔씨.   
1  10-2022-0151974                   디어젠 주식회사   
2  10-2025-7023914                 보레알리스 게엠베하   
3  10-2022-7040957  스트롱 포스 티피 포트폴리오 2022, 엘엘씨   
4  10-2023-0090053   한동대학교 산학협력단, 주식회사 이랜드리테일   

                                                                                                                                                                                                                                                                                                                          IPC코드  \
0  G16H 50/30, G16H 50/20, G16H 50/50, G16H 30/20, G06T 7/00, G06T 7/40, G06T 7/62, G06T 7/90, A61B 5/00, A61B 5/00, A61B 5/00, A61B 5/00, G16H 15/00, G16H 20/30, G16H 20/60, G16H 20/70, G16H 40/67, G06N 3/0464, G06N 3/045, G06N 3/044, G06N 20/20, G06N 20/10, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/096, G06N 3/0985   
1                                                                                                                                                                 G16C 20/70, G16C 20/30, G16C 20/40, G06N 3/045, G06N 3/0985, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/092, G06N 3/0464, G06N 3/044, G06N 3/0475, G06N 3/094   
2                                                                                                                                                                               G16C 60/00, G16C 20/70, G16C 20/30, G06N 3/0499, G06N 3/045, G06N 3/0985, G06N 3/048, G06N 3/088, G06N 3/09, G06N 3/092, G06N 3/044, G06N 3/086   
3                                                                                                          G06F 11/34, G06F 11/34, G06F 11/30, G06F 11/30, G06F 11/30, G06F 30/27, G06F 17/18, G06N 20/10, G06N 3/006, G06N 3/042, G06N 3/044, G06N 3/045, G06N 3/08, G06N 3/126, G06N 7/01, G06Q 50/40, G06F 30/15, G06F 30/25   
4                                                                                                                G16H 20/60, G16H 10/60, G16H 10/20, G16H 50/20, G16H 50/70, G16H 50/50, G06Q 30/0601, G06N 3/045, G06N 20/00, G06N 3/0464, G06N 3/044, G06N 3/0475, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/092, G06N 3/084   

                                                         발명의 명칭  \
0                                                      동물 건강 평가   
1                                    신경망 모델을 활용하여 분자를 최적화시키는 방법   
2  하나 이상의 물질의 적어도 탄소성 기계적 응답을 나타내는 파라미터 값을 출력하기 위한 컴퓨터로 구현되는 방법   
3                                       운송 시스템용 디지털 트윈 시스템 및 방법   
4                          인공지능 기반의 건강상태 맞춤 식단 및 상품 추천 방법 및 시스템   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [34]:
df

출원번호                        출원인  \
0  10-2022-7003979                올리 펫츠 아이엔씨.   
1  10-2022-0151974                   디어젠 주식회사   
2  10-2025-7023914                 보레알리스 게엠베하   
3  10-2022-7040957  스트롱 포스 티피 포트폴리오 2022, 엘엘씨   
4  10-2023-0090053   한동대학교 산학협력단, 주식회사 이랜드리테일   

                                                                                                                                                                                                                                                                                                                          IPC코드  \
0  G16H 50/30, G16H 50/20, G16H 50/50, G16H 30/20, G06T 7/00, G06T 7/40, G06T 7/62, G06T 7/90, A61B 5/00, A61B 5/00, A61B 5/00, A61B 5/00, G16H 15/00, G16H 20/30, G16H 20/60, G16H 20/70, G16H 40/67, G06N 3/0464, G06N 3/045, G06N 3/044, G06N 20/20, G06N 20/10, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/096, G06N 3/0985   
1                                                                                                                                                                 G16C 20/70, G16C 20/30, G16C 20/40, G06N 3/045, G06N 3/0985, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/092, G06N 3/0464, G06N 3/044, G06N 3/0475, G06N 3/094   
2                                                                                                                                                                               G16C 60/00, G16C 20/70, G16C 20/30, G06N 3/0499, G06N 3/045, G06N 3/0985, G06N 3/048, G06N 3/088, G06N 3/09, G06N 3/092, G06N 3/044, G06N 3/086   
3                                                                                                          G06F 11/34, G06F 11/34, G06F 11/30, G06F 11/30, G06F 11/30, G06F 30/27, G06F 17/18, G06N 20/10, G06N 3/006, G06N 3/042, G06N 3/044, G06N 3/045, G06N 3/08, G06N 3/126, G06N 7/01, G06Q 50/40, G06F 30/15, G06F 30/25   
4                                                                                                                G16H 20/60, G16H 10/60, G16H 10/20, G16H 50/20, G16H 50/70, G16H 50/50, G06Q 30/0601, G06N 3/045, G06N 20/00, G06N 3/0464, G06N 3/044, G06N 3/0475, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/092, G06N 3/084   

                                                         발명의 명칭  \
0                                                      동물 건강 평가   
1                                    신경망 모델을 활용하여 분자를 최적화시키는 방법   
2  하나 이상의 물질의 적어도 탄소성 기계적 응답을 나타내는 파라미터 값을 출력하기 위한 컴퓨터로 구현되는 방법   
3                                       운송 시스템용 디지털 트윈 시스템 및 방법   
4                          인공지능 기반의 건강상태 맞춤 식단 및 상품 추천 방법 및 시스템   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [30]:
def fetch_pdf_and_drawing(app_num, api_key):
    """특정 출원번호의 공개전문 PDF와 대표도면 URL을 가져옵니다."""
    
    # 1. 공개전문 PDF URL 조회
    pdf_url_api = "http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getPubFullTextInfoSearch"
    pdf_params = {'applicationNumber': app_num, 'ServiceKey': api_key}
    pdf_link = "없음"
    
    try:
        res_pdf = requests.get(pdf_url_api, params=pdf_params)
        if res_pdf.status_code == 200:
            root_pdf = ET.fromstring(res_pdf.text)
            # body/item/path 태그에서 PDF 링크 추출
            pdf_path = root_pdf.findtext('.//body/item/path')
            if pdf_path:
                pdf_link = pdf_path
    except Exception as e:
        pdf_link = f"에러: {e}"

    # 2. 대표도면 URL 조회
    drawing_url_api = "http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getReprsntFloorPlanInfoSearch"
    drawing_params = {'applicationNumber': app_num, 'ServiceKey': api_key}
    drawing_link = "없음"
    
    try:
        res_draw = requests.get(drawing_url_api, params=drawing_params)
        if res_draw.status_code == 200:
            root_draw = ET.fromstring(res_draw.text)
            # body/imagePathInfo 에서 largePath 우선 추출, 없으면 path 추출
            large_path = root_draw.findtext('.//body/imagePathInfo/largePath')
            normal_path = root_draw.findtext('.//body/imagePathInfo/path')
            
            if large_path:
                drawing_link = large_path
            elif normal_path:
                drawing_link = normal_path
    except Exception as e:
        drawing_link = f"에러: {e}"

    return {
        "출원번호": app_num,
        "공개전문 PDF": pdf_link,
        "대표도면 URL": drawing_link
    }

In [33]:
df.drop('도면URL', axis=1, inplace=True)

In [31]:
results = []
print("PDF 및 대표도면 링크 수집을 시작합니다...")

for num in APP_NUMBERS:
    print(f"[{num}] 조회 중...")
    data = fetch_pdf_and_drawing(num, API_KEY)
    results.append(data)
    time.sleep(1)  # API 두 개를 연속 호출하므로 서버 부하 방지를 위해 1초 대기

print("데이터 수집 완료!\n")

PDF 및 대표도면 링크 수집을 시작합니다...
[1020227003979] 조회 중...
[1020220151974] 조회 중...
[1020257023914] 조회 중...
[1020227040957] 조회 중...
[1020230090053] 조회 중...
데이터 수집 완료!



In [37]:
df_media = pd.DataFrame(results)
df_media

,출원번호,공개전문 PDF,대표도면 URL
0,1020227003979,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=12df679b84029f739813e9e1875bb857bef61bd56bbd0562cd8386e85004e07cf6123ee3349d7730c3ce59b395c87d250fa781308dcd9dd95aee586d02f9de6badece06fb5288f93,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=6c650beb4cee9ce4122b704b88878c93221179325cbd088b4e40354685132771ac3be1da2970e1f9e853dac8ee5c20fdfb25b35a92847e411c9b4a3c01542eabd0235897a7624155
1,1020220151974,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=11ea9f5dfe90b683899065ac5d8b8e2629c6136118273fde74f9813ace3dd512375a26f81004d26a169dc2b0b1765ee2f76e532ca50c57a6cf9625aba1fe35fc2cd9a516d6b1adb3,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=6c650beb4cee9ce4122b704b88878c93a22c639832dc47dcab21ea31cbffe36f2472f0b87a245eeedbaf143fc836973788de6d3d84723abe91f1e6d0b63f362ddc512a716ddab081
2,1020257023914,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=11ea9f5dfe90b683899065ac5d8b8e26f307660c7d8fa8327f51a2ac46b9345f38b1602d9ef99ade113628f1a967d1d6e2bc2b2a3904af222a365166d630ab01f7107558e3060316,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=6c650beb4cee9ce4122b704b88878c9330405d1f9677e096c1078880360179143fa3131bdd187bdaac5a2ffd1bb9e443a1b55f633568c21855da619b763af13491dc88b0ac8bcc90
3,1020227040957,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=12df679b84029f739813e9e1875bb8570d62618c77cf85194e7c107d7ea4cb035afcbf90faf9c63da22ea181a6051d71e183db90e8a3dc8e1c8eb16cc6ae32270f53d4fd1b3d8cb3,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=6c650beb4cee9ce4122b704b88878c9341339e1f938b233bba80425f2255c37328ddd2d39644d543c6579f8166d6d1570cbae03aa88d428627bb6254480328b6128e42f1f26cb1cf
4,1020230090053,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=11ea9f5dfe90b683899065ac5d8b8e26f307660c7d8fa832631f3b1f32ab3e6682bf17e8da367b1ceaf5d15975198cffcebf87b35ff1a727bb09659d15ffbb0a9cbb052531605f38,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=6c650beb4cee9ce4122b704b88878c93101a960faad0067ea421804e799fafeff3415d87dab32b4586415debadc72d682a29abc505b6900a85ecbd0300d67b777565f96f2184a0dd


In [35]:
df_total = pd.merge(df, df_media, on='출원번호', how='left')

In [36]:
df_total.head()

출원번호                        출원인  \
0  10-2022-7003979                올리 펫츠 아이엔씨.   
1  10-2022-0151974                   디어젠 주식회사   
2  10-2025-7023914                 보레알리스 게엠베하   
3  10-2022-7040957  스트롱 포스 티피 포트폴리오 2022, 엘엘씨   
4  10-2023-0090053   한동대학교 산학협력단, 주식회사 이랜드리테일   

                                                                                                                                                                                                                                                                                                                          IPC코드  \
0  G16H 50/30, G16H 50/20, G16H 50/50, G16H 30/20, G06T 7/00, G06T 7/40, G06T 7/62, G06T 7/90, A61B 5/00, A61B 5/00, A61B 5/00, A61B 5/00, G16H 15/00, G16H 20/30, G16H 20/60, G16H 20/70, G16H 40/67, G06N 3/0464, G06N 3/045, G06N 3/044, G06N 20/20, G06N 20/10, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/096, G06N 3/0985   
1                                                                                                                                                                 G16C 20/70, G16C 20/30, G16C 20/40, G06N 3/045, G06N 3/0985, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/092, G06N 3/0464, G06N 3/044, G06N 3/0475, G06N 3/094   
2                                                                                                                                                                               G16C 60/00, G16C 20/70, G16C 20/30, G06N 3/0499, G06N 3/045, G06N 3/0985, G06N 3/048, G06N 3/088, G06N 3/09, G06N 3/092, G06N 3/044, G06N 3/086   
3                                                                                                          G06F 11/34, G06F 11/34, G06F 11/30, G06F 11/30, G06F 11/30, G06F 30/27, G06F 17/18, G06N 20/10, G06N 3/006, G06N 3/042, G06N 3/044, G06N 3/045, G06N 3/08, G06N 3/126, G06N 7/01, G06Q 50/40, G06F 30/15, G06F 30/25   
4                                                                                                                G16H 20/60, G16H 10/60, G16H 10/20, G16H 50/20, G16H 50/70, G16H 50/50, G06Q 30/0601, G06N 3/045, G06N 20/00, G06N 3/0464, G06N 3/044, G06N 3/0475, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/092, G06N 3/084   

                                                         발명의 명칭  \
0                                                      동물 건강 평가   
1                                    신경망 모델을 활용하여 분자를 최적화시키는 방법   
2  하나 이상의 물질의 적어도 탄소성 기계적 응답을 나타내는 파라미터 값을 출력하기 위한 컴퓨터로 구현되는 방법   
3                                       운송 시스템용 디지털 트윈 시스템 및 방법   
4                          인공지능 기반의 건강상태 맞춤 식단 및 상품 추천 방법 및 시스템   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [38]:
!pip install sqlalchemy psycopg2-binary

In [39]:
df['출원번호'] = df['출원번호'].str.replace('-', '')

In [41]:
df_total = pd.merge(df, df_media, on='출원번호', how='left')

In [42]:
df_total.head()

출원번호                        출원인  \
0  1020227003979                올리 펫츠 아이엔씨.   
1  1020220151974                   디어젠 주식회사   
2  1020257023914                 보레알리스 게엠베하   
3  1020227040957  스트롱 포스 티피 포트폴리오 2022, 엘엘씨   
4  1020230090053   한동대학교 산학협력단, 주식회사 이랜드리테일   

                                                                                                                                                                                                                                                                                                                          IPC코드  \
0  G16H 50/30, G16H 50/20, G16H 50/50, G16H 30/20, G06T 7/00, G06T 7/40, G06T 7/62, G06T 7/90, A61B 5/00, A61B 5/00, A61B 5/00, A61B 5/00, G16H 15/00, G16H 20/30, G16H 20/60, G16H 20/70, G16H 40/67, G06N 3/0464, G06N 3/045, G06N 3/044, G06N 20/20, G06N 20/10, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/096, G06N 3/0985   
1                                                                                                                                                                 G16C 20/70, G16C 20/30, G16C 20/40, G06N 3/045, G06N 3/0985, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/092, G06N 3/0464, G06N 3/044, G06N 3/0475, G06N 3/094   
2                                                                                                                                                                               G16C 60/00, G16C 20/70, G16C 20/30, G06N 3/0499, G06N 3/045, G06N 3/0985, G06N 3/048, G06N 3/088, G06N 3/09, G06N 3/092, G06N 3/044, G06N 3/086   
3                                                                                                          G06F 11/34, G06F 11/34, G06F 11/30, G06F 11/30, G06F 11/30, G06F 30/27, G06F 17/18, G06N 20/10, G06N 3/006, G06N 3/042, G06N 3/044, G06N 3/045, G06N 3/08, G06N 3/126, G06N 7/01, G06Q 50/40, G06F 30/15, G06F 30/25   
4                                                                                                                G16H 20/60, G16H 10/60, G16H 10/20, G16H 50/20, G16H 50/70, G16H 50/50, G06Q 30/0601, G06N 3/045, G06N 20/00, G06N 3/0464, G06N 3/044, G06N 3/0475, G06N 3/09, G06N 3/088, G06N 3/0895, G06N 3/092, G06N 3/084   

                                                         발명의 명칭  \
0                                                      동물 건강 평가   
1                                    신경망 모델을 활용하여 분자를 최적화시키는 방법   
2  하나 이상의 물질의 적어도 탄소성 기계적 응답을 나타내는 파라미터 값을 출력하기 위한 컴퓨터로 구현되는 방법   
3                                       운송 시스템용 디지털 트윈 시스템 및 방법   
4                          인공지능 기반의 건강상태 맞춤 식단 및 상품 추천 방법 및 시스템   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [43]:
import pandas as pd
from sqlalchemy import create_engine

In [44]:
df_total.to_csv('upload_data.csv', index=False)

In [51]:
df_total.drop('대표도면 URL', axis=1, inplace=True)

In [52]:
df_total

,공개전문 PDF
0,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=12df679b84029f739813e9e1875bb857bef61bd56bbd0562cd8386e85004e07cf6123ee3349d7730c3ce59b395c87d250fa781308dcd9dd95aee586d02f9de6badece06fb5288f93
1,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=11ea9f5dfe90b683899065ac5d8b8e2629c6136118273fde74f9813ace3dd512375a26f81004d26a169dc2b0b1765ee2f76e532ca50c57a6cf9625aba1fe35fc2cd9a516d6b1adb3
2,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=11ea9f5dfe90b683899065ac5d8b8e26f307660c7d8fa8327f51a2ac46b9345f38b1602d9ef99ade113628f1a967d1d6e2bc2b2a3904af222a365166d630ab01f7107558e3060316
3,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=12df679b84029f739813e9e1875bb8570d62618c77cf85194e7c107d7ea4cb035afcbf90faf9c63da22ea181a6051d71e183db90e8a3dc8e1c8eb16cc6ae32270f53d4fd1b3d8cb3
4,http://plus.kipris.or.kr/openapi/fileToss.jsp?arg=11ea9f5dfe90b683899065ac5d8b8e26f307660c7d8fa832631f3b1f32ab3e6682bf17e8da367b1ceaf5d15975198cffcebf87b35ff1a727bb09659d15ffbb0a9cbb052531605f38


In [53]:
df_total.to_csv('test.csv', index=False)